# External link use

**Does an opening with an external link win more often?** Score the original Markdown in the published training pairs. The binary feature flags at least one HTTP(S) URL or `www.` address with an external hostname. The rule counts inline Markdown targets and bare URLs, skips link-label text, code, quoted lines, and marked edit sections, and excludes Reddit domains and their subdomains.

An outbound URL can point to many kinds of material. The feature measures link presence and treats all included destinations alike. The [feature script](analyze_external_links.py) saves opening and pair scores. The [model script](../models/train_p2_plus_reply_timing_and_external_link.py) adds the A-minus-B indicator difference to P2 with continuous reply delay.

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "team-setup").is_dir() and (p / "pyproject.toml").is_file()
)
ROOT = REPO / "project/data/processed/linguistic-features"
MODELS = ROOT / "paired-models"
MODEL_RUN_ID = "20261120T220731301773Z"
model_run = MODELS / "p2_plus_reply_timing_and_external_link" / MODEL_RUN_ID
model_report = json.loads((model_run / "results.json").read_text())
link_run = ROOT / "external-links" / model_report["link_input_run"]
report = json.loads((link_run / "results.json").read_text())
pair_path = link_run / "paired_scores.parquet"
assert report["split"] == model_report["split"] == "train"
assert hashlib.sha256(pair_path.read_bytes()).hexdigest() == model_report["link_input_sha256"]
pairs = pd.read_parquet(pair_path)
openings = pd.read_parquet(link_run / "opening_scores.parquet")
assert pairs.groupby("post_id")["fold"].nunique().eq(1).all()
print(f"{len(pairs):,} training pairs across {report['discussions']:,} OP discussions")
print(f"{report['openings_with_external_link']:,} of {len(openings):,} distinct openings contain an external link.")

3,451 training pairs across 2,507 OP discussions
944 of 6,581 distinct openings contain an external link.


## Archive examples and paired signal

Inspect the first link in eight seeded example openings. The examples include an encyclopedia, a university news site, and general-interest sites. Examples use the archived text and hostname.

The win rate is calculated among pairs where exactly one side links out. Report the results across all pairs and among replies with word counts within 50 words, with discussion-bootstrap intervals. The standalone model uses only the binary feature difference.

In [2]:
linked = openings.loc[openings["has_external_link"].eq(1)]
display(linked.sample(min(8, len(linked)), random_state=42)[[
    "first_external_host", "first_external_link_context",
]])
print(f"Both openings link in {report['both_link']:,} pairs.")
print(f"Neither opening links in {report['neither_links']:,} pairs.")
rows = []
for label, key in (("All pairs", "all_pairs"), ("Within 50 words", "within_50_words")):
    counts = report[key]
    low, high = counts["ci_95"]
    rows.append({
        "Subset": label, "Pairs": counts["pairs"], "Exactly one links": counts["pairs"] - counts["ties"],
        "Linked opening won": counts["winner_higher"], "Linked opening won (%)": 100 * counts["winner_higher_rate"],
        "95% CI lower (%)": 100 * low, "95% CI upper (%)": 100 * high,
    })
display(pd.DataFrame(rows).set_index("Subset").round(2))
display(pd.Series({
    "ROC-AUC": report["prediction"]["roc_auc"],
    "Average precision": report["prediction"]["average_precision"],
}, name="Link-only out-of-fold score").round(4).to_frame())

,first_external_host,first_external_link_context
comment_id,,
t1_cl7q1dy,en.wikipedia.org,http://en.wikipedia.org/wiki/Public_good
t1_ci34zl5,en.wikipedia.org,This gets extremely confusing. If you read [this article](http://en.wikipedia.org/wiki/Rights_and_responsibilities_of_marriages_in_the_United_States) you'll notice that a great many of the responsibilities within are designed to make othe
t1_cqw2c9f,www.911myths.com,"y the SEC and FBI following 9/11, when they both looked into the suspicous trades, [here](http://www.911myths.com/index.php/Put_Options). It's a long read so I'm going to pull out some of the highlights:"
t1_cn4ba9v,freakonomics.com,not quite related but there was a [freakonomics article](http://freakonomics.com/2012/07/11/whats-wrong-with-cash-for-grades-a-new-marketplace-podcast/) about paying high schoolers for passing grades. I don't think it worked. EDIT: Bits o
t1_cncxwgs,www.eatbydate.com,http://www.eatbydate.com/
t1_cifodcl,www.examiner.com,"r dropped him the final time, Taker had his bell rung a bit and [legit couldn't kick out](http://www.examiner.com/article/wwe-rumors-undertaker-legitimately-hurt-by-brock-lesnar-at-wrestlemania-30). He struggled to get up (no one came to help them because he's known to fell it), except"
t1_cc5on04,news.harvard.edu,"despised, including several [armed student insurrections as recently as 1834 at Harvard](http://news.harvard.edu/gazette/story/2012/04/harvards-long-ago-student-risings/)."
t1_chzyi89,www.webmd.com,http://www.webmd.com/depression/guide/depression-symptoms-and-types


Both openings link in 123 pairs.
Neither opening links in 2,595 pairs.


,Pairs,Exactly one links,Linked opening won,Linked opening won (%),95% CI lower (%),95% CI upper (%)
Subset,,,,,,
All pairs,3451,733,506,69.03,65.66,72.48
Within 50 words,1144,189,135,71.43,64.55,77.13


,Link-only out-of-fold score
ROC-AUC,0.5689
Average precision,0.5544


## Does link use add value beyond P2 and delay?

Compare the same three OP-discussion-grouped training folds. Each model fits standardization inside the fold. The 95% intervals use 1,000 paired bootstrap draws by OP discussion. The held-out pairs are outside this experiment.

In [3]:
prediction = pd.read_parquet(model_run / "oof_predictions.parquet")
baseline_path = MODELS / "p2_plus_reply_timing" / model_report["baseline_run"] / "oof_predictions.parquet"
assert hashlib.sha256(baseline_path.read_bytes()).hexdigest() == model_report["baseline_predictions_sha256"]
baseline = pd.read_parquet(baseline_path)
keys = ["pair_id", "post_id", "a_comment_id", "b_comment_id", "a_wins", "fold"]
feature = "a_minus_b_has_external_link"
pd.testing.assert_frame_equal(prediction[keys], baseline[keys])
pd.testing.assert_frame_equal(prediction[[*keys, feature]], pairs[[*keys, feature]])
np.testing.assert_array_equal(prediction["p2_plus_reply_timing_oof_probability_a_wins"],
                              baseline["oof_probability_a_wins"])

model_labels = ["P2 + reply timing", "P2 + reply timing + link"]
data = prediction.assign(**{
    model_labels[0]: prediction["p2_plus_reply_timing_oof_probability_a_wins"],
    model_labels[1]: prediction["oof_probability_a_wins"],
})

In [4]:
from sklearn.metrics import accuracy_score, average_precision_score, log_loss, roc_auc_score


def pairwise_accuracy(y, probability, sample_weight=None):
    return accuracy_score(y, probability >= 0.5, sample_weight=sample_weight)


metric_functions = {
    "ROC-AUC": roc_auc_score,
    "Average precision": average_precision_score,
    "Pairwise accuracy": pairwise_accuracy,
    "Log loss": log_loss,
}
y = data["a_wins"].to_numpy()
p2, expanded_probability = data[model_labels].to_numpy().T
post_codes, posts = pd.factorize(data["post_id"])
rng = np.random.default_rng(42)
differences = {name: [] for name in metric_functions}
for _ in range(1000):
    post_weights = rng.multinomial(len(posts), np.full(len(posts), 1 / len(posts)))
    weights = post_weights[post_codes]
    for name, metric in metric_functions.items():
        differences[name].append(
            metric(y, expanded_probability, sample_weight=weights)
            - metric(y, p2, sample_weight=weights)
        )

rows = []
for name, metric in metric_functions.items():
    base, extra = metric(y, p2), metric(y, expanded_probability)
    low, high = np.quantile(differences[name], [0.025, 0.975])
    rows.append({
        "Metric": name, model_labels[0]: base, model_labels[1]: extra, "Change": extra - base,
        "95% CI lower": low, "95% CI upper": high,
    })
performance = pd.DataFrame(rows).set_index("Metric")
display(performance.round(5))

,P2 + reply timing,P2 + reply timing + link,Change,95% CI lower,95% CI upper
Metric,,,,,
ROC-AUC,0.72380,0.73125,0.00745,0.00198,0.01292
Average precision,0.70948,0.71836,0.00888,0.00284,0.01482
Pairwise accuracy,0.66416,0.66734,0.00319,-0.00376,0.01046
Log loss,0.61415,0.60787,-0.00628,-0.01053,-0.00252
